# BOI v1.16 — HNSW Backend (Best QPS)

Thin config + glue notebook. All pipeline logic lives in `boi_pipeline.py`.
Only the per-backend HNSW build worker is defined here.

In [1]:
# ============================================================
# CONFIG — all tunable knobs live here
# ============================================================
import os

# ── Dataset ──────────────────────────────────────────────────
ENCODING_DIR         = "/raid/ssEncodingData/encoding/pk-real0.002"
ENCODING_FILE_PREFIX = "real_"
GROUND_TRUTH_DIR     = "/raid/ssEncodingData/warehouse/pk-query-187019"

TRAIN_RATIO          = 0.80   # fraction used as DB
MAX_QUERIES          = None   # None = use all
ZERO_TOL             = 1e-12

# ── Shared cache (encodings, GT, flat arrays, signatures) ────
_dataset_key     = os.path.basename(ENCODING_DIR.rstrip("/"))
SHARED_CACHE_DIR = (
    f"/raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/{_dataset_key}/"
)

# ── Signatures ───────────────────────────────────────────────
L        = 100   # total MinHash components
WMH_SEED = 42

# ── Banding ──────────────────────────────────────────────────
B = 50           # number of bands
R = 2            # rows per band  (must satisfy L == B*R)

# ── Rebalance ────────────────────────────────────────────────
REB_MERGE_SMALL = 0
REB_SPLIT_LARGE = 3000

# ── HNSW build ───────────────────────────────────────────────
BUCKET_INDEX_MIN_SIZE = 1000  # tau: smaller buckets → exact-only
BUILD_WORKERS         = 60
NUM_THREADS           = 4     # threads per HNSW build worker
LOCAL_HNSW_M          = 16
LOCAL_HNSW_EFC        = 150
LOCAL_HNSW_EFS        = 400

# ── Bucket index cache directory ─────────────────────────────
_INDEX_CACHE_KEY = (
    f"L{L}_B{B}_R{R}"
    f"_M{LOCAL_HNSW_M}_efc{LOCAL_HNSW_EFC}"
    f"_minSz{BUCKET_INDEX_MIN_SIZE}"
    f"_rebSplit{REB_SPLIT_LARGE}"
    f"_sigvec"
)
BUCKET_CACHE_DIR = os.path.join(
    "/raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k",
    _dataset_key,
    f"bucket_indexes_{_INDEX_CACHE_KEY}",
)

# ── Query ─────────────────────────────────────────────────────
FAST_TOPK          = 500
FAST_ALPHA_MULT    = 0.08
FAST_EPSILON       = 4000
FAST_EFS           = 400
FAST_HNSW_K_MULT   = 1
QUERY_WORKERS      = 150
CHUNKS_PER_WORKER  = 4
MAX_BANDS_TO_PROBE = 40

# ── Encoding load ─────────────────────────────────────────────
ENC_LOAD_WORKERS = 32

print("Config ready.")
print(f"  dataset_key      : {_dataset_key}")
print(f"  SHARED_CACHE_DIR : {SHARED_CACHE_DIR}")
print(f"  BUCKET_CACHE_DIR : {BUCKET_CACHE_DIR}")


Config ready.
  dataset_key      : pk-real0.002
  SHARED_CACHE_DIR : /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/
  BUCKET_CACHE_DIR : /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/bucket_indexes_L100_B50_R2_M16_efc150_minSz1000_rebSplit3000_sigvec


In [2]:
import sys
sys.path.insert(0, "/raid/ruban/boi_multi_index")
import boi_pipeline as bp

import numpy as np, gc, os, time

# Clear stale shared-memory blocks from prior runs
bp.release_stale_shm()
bp.print_memory_stats("After startup")


Not found (clean): boi_ids_flat
Not found (clean): boi_vals_flat
Not found (clean): boi_offsets
Not found (clean): boi_sig_all
[After startup] RSS=0.14 GB | Avail=1943.00/2015.68 GB | Used=3.6%


In [3]:
encodings, DATA_END, QUERY_START, QUERY_END, POLY_COUNT = bp.load_encodings_cached(
    encoding_dir=ENCODING_DIR,
    cache_dir=SHARED_CACHE_DIR,
    train_ratio=TRAIN_RATIO,
    max_queries=MAX_QUERIES,
    file_prefix=ENCODING_FILE_PREFIX,
    zero_tol=ZERO_TOL,
    num_workers=ENC_LOAD_WORKERS,
)

gt_neighbors, gt_offsets, gt_qids, gt_qid_map = bp.load_ground_truth_cached(
    gt_dir=GROUND_TRUTH_DIR,
    cache_dir=SHARED_CACHE_DIR,
)

# Dimensionality — needed for query worker init
D = max(int(ids.max()) for ids, _ in encodings if ids.size > 0) + 1
print(f"D={D:,}")


[Cache hit] Loading encodings from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/encodings_payload.pkl
Cached encodings loaded in 4.65s
POLY_COUNT=233,773 | DB=[0,187019) | Q=[187019,233773)
[After encoding load] RSS=9.07 GB | Avail=1934.08/2015.68 GB | Used=4.0%
[Cache hit] Loading GT from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/gt_csr.pkl
D=18,219


In [4]:
ids_flat, vals_flat, offsets = bp.build_flat_encoding_arrays_cached(
    encodings=encodings,
    cache_dir=SHARED_CACHE_DIR,
)

sig_all = bp.compute_signatures_cached(
    encodings=encodings,
    ids_flat=ids_flat, vals_flat=vals_flat, offsets=offsets,
    L=L, seed=WMH_SEED,
    cache_dir=SHARED_CACHE_DIR,
)

del encodings; gc.collect()
bp.print_memory_stats("After signature compute")

SHM_META, _SHM_REGISTRY = bp.setup_shared_memory(ids_flat, vals_flat, offsets, sig_all)


[Cache hit] Loading flat arrays from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/flat_encoding_arrays.pkl
[Cache hit] WMH params from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/wmh_params_L100.pkl
Numba WMH warmup...
Warmup done.
[Cache hit] Signatures from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/sig_all_L100.npy
sig_all: shape=(233773, 100) dtype=float32
[After signatures] RSS=18.58 GB | Avail=1924.59/2015.68 GB | Used=4.5%
[After signature compute] RSS=18.52 GB | Avail=1924.65/2015.68 GB | Used=4.5%
Moving arrays to shared memory...
Shared memory ready in 3.51s
[After shared memory setup] RSS=27.16 GB | Avail=1916.00/2015.68 GB | Used=4.9%


In [5]:
banding = bp.Banding(B=B, R=R)
assert L == B * R

sig_data = sig_all[:DATA_END]

part_map_raw, _ = bp.build_partition_map(sig_data, banding)
part_map, rebalance_stats = bp.rebalance_part_map(
    part_map_raw, sig_data, R=banding.R,
    merge_small_threshold=REB_MERGE_SMALL,
    split_large_threshold=REB_SPLIT_LARGE,
)
del part_map_raw; gc.collect()
print("Rebalance stats:")
for k, v in rebalance_stats.items():
    print(f"  {k}: {v}")


Building band partitions: 100%|██████████| 187019/187019 [00:14<00:00, 12801.73it/s]


Rebalance stats:
  total_buckets: 207902
  mean_bucket_size: 44.97768179238295
  median_bucket_size: 2.0
  max_bucket_size: 34095
  p90_bucket_size: 35.0


In [6]:
# ── HNSW build worker (v1.16-specific) ───────────────────────
import uuid

def _build_one_bucket_hnsw(args):
    """
    Per-bucket HNSW build worker.
    Uses bp.build_hnsw_index (Buddhi's createIndex pattern) on 100-dim WMH signatures.
    """
    bucket_key, member_ids = args
    t0 = time.time()
    G  = bp._BUCKET_BUILD_GLOBALS

    X   = G["sig_all"][member_ids].astype("float32")
    idx = bp.build_hnsw_index(
        X,
        M=G["local_hnsw_m"],
        efC=G["local_hnsw_efc"],
        num_threads=G["num_threads"],
        post=1,
    )
    idx.setQueryTimeParams({"efSearch": G["local_hnsw_efs"]})

    path = os.path.join(G["tmp_index_dir"], f"bucket_{uuid.uuid4().hex}")
    bp.save_hnsw_index(idx, path)

    return {
        "bucket_key":         bucket_key,
        "member_ids":         member_ids,
        "size":               int(member_ids.size),
        "use_index":          True,
        "index_path":         path,
        "worker_elapsed_sec": time.time() - t0,
    }


In [7]:
bucket_indexes, split_index, build_stats = bp.build_or_load_bucket_indexes(
    part_map=part_map,
    worker_fn=_build_one_bucket_hnsw,
    shm_meta=SHM_META,
    cache_dir=BUCKET_CACHE_DIR,
    min_bucket_size=BUCKET_INDEX_MIN_SIZE,
    build_workers=BUILD_WORKERS,
    extra_worker_kwargs={
        "local_hnsw_m":   LOCAL_HNSW_M,
        "local_hnsw_efc": LOCAL_HNSW_EFC,
        "local_hnsw_efs": LOCAL_HNSW_EFS,
        "num_threads":    NUM_THREADS,
    },
    local_hnsw_efs=LOCAL_HNSW_EFS,
)
print("Build stats:")
for k, v in build_stats.items():
    print(f"  {k}: {v}")


[Cache hit] Loading bucket indexes from /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache_233k/pk-real0.002/bucket_indexes_L100_B50_R2_M16_efc150_minSz1000_rebSplit3000_sigvec


Loading bucket indexes: 100%|██████████| 207902/207902 [00:04<00:00, 44604.02it/s] 

Loaded 207,902 buckets (2,171 indexed, 205,731 exact)
Build stats:
  total_buckets: 207902
  indexed_buckets: 2171
  exact_only_buckets: 205731
  total_members_indexed: 5149371
  mean_bucket_size: 44.97768179238295
  median_bucket_size: 2.0
  max_bucket_size: 34095
  build_time_sec: 25.306350708007812
  avg_worker_build_sec: 0.46207147048728797
  avg_reload_sec: 0.00676869775006871


In [8]:
results_dict, logs_dict, summary, pre_rerank_dict = bp.query_boi_weighted_parallel_processpool(
    query_ids        = list(range(QUERY_START, QUERY_END)),
    banding          = banding,
    bucket_indexes   = bucket_indexes,
    split_index      = split_index,
    shm_meta         = SHM_META,
    data_end         = DATA_END,
    query_start      = QUERY_START,
    D                = D,
    topK             = FAST_TOPK,
    alpha_mult       = FAST_ALPHA_MULT,
    epsilon          = FAST_EPSILON,
    efS              = FAST_EFS,
    max_bands_to_probe = MAX_BANDS_TO_PROBE,
    num_workers      = QUERY_WORKERS,
    chunks_per_worker= CHUNKS_PER_WORKER,
    return_timing    = True,
    compute_pre_rerank = True,
    use_batch          = True,
)



BOI query [batch] | n=46754 | workers=150 | chunks=600 | topK=500 | alpha_mult=0.08 | epsilon=4000


Query chunks: 100%|██████████| 600/600 [02:26<00:00,  4.10it/s]


QPS=261.48 | Total=178.81s


In [9]:
print("=== Recall (before rerank) ===")
recall_pre = bp.evaluate_results(
    pre_rerank_dict, gt_neighbors, gt_offsets, gt_qid_map, ks=(10, 50, 500),
)
for k, v in recall_pre.items():
    print(f"  R@{k} : {v:.4f}" if v is not None else f"  R@{k} : N/A")

print("\n=== Recall (after rerank) ===")
recall = bp.evaluate_results(
    results_dict, gt_neighbors, gt_offsets, gt_qid_map, ks=(10, 50, 500),
)
for k, v in recall.items():
    print(f"  R@{k} : {v:.4f}" if v is not None else f"  R@{k} : N/A")
print(f"  QPS  : {summary['qps']:.2f}")

gt_dict = bp.read_ground_truth_list(GROUND_TRUTH_DIR)
print("\n=== Recall Buddhi formula (after rerank) ===")
buddhi_recall = bp.buddhi_recall_from_boi_results(results_dict, gt_dict, K_list=(10, 50, 500))

print("\n=== Timing breakdown (per-query averages) ===")
t = summary
rows = [
    ("Stage",               "ms / query"),
    ("─"*28,               "─"*12),
    ("HNSW probe",          f"{t.get('avg_hnsw_probe_sec',0)*1000:.2f}"),
    ("Exact probe",         f"{t.get('avg_exact_probe_sec',0)*1000:.2f}"),
    ("  └─ total probe",    f"{t.get('avg_probe_sec',0)*1000:.2f}"),
    ("Prune (alpha+eps)",   f"{t.get('avg_prune_sec',0)*1000:.2f}"),
    ("Rerank (exact WJ)",   f"{t.get('avg_rerank_sec',0)*1000:.2f}"),
    ("Total per query",     f"{t.get('avg_total_sec',0)*1000:.2f}"),
]
for label, val in rows:
    print(f"  {label:<28} {val:>12}")

print("\n=== Candidate stats (per-query averages) ===")
stats = [
    ("HNSW buckets hit",       t.get('avg_indexed_buckets_hit',0)),
    ("Exact buckets hit",      t.get('avg_exact_buckets_hit',0)),
    ("HNSW candidates added",  t.get('avg_indexed_candidates_added',0)),
    ("Exact postings scanned", t.get('avg_postings_scanned',0)),
]
for label, val in stats:
    print(f"  {label:<30} {val:>10.1f}")
print(f"  {'QPS':<30} {t['qps']:>10.2f}")


=== Recall (before rerank) ===
[evaluate_results] Skipped 2088 queries with no GT entry.
  R@10 : 0.5069
  R@50 : 0.5929
  R@500 : 0.7106

=== Recall (after rerank) ===
[evaluate_results] Skipped 2088 queries with no GT entry.
  R@10 : 0.9885
  R@50 : 0.9821
  R@500 : 0.8869
  QPS  : 261.48

=== Recall Buddhi formula (after rerank) ===
  R@10: full=0.9444  adj=0.9885
  R@50: full=0.9383  adj=0.9821
  R@500: full=0.8473  adj=0.8869

=== Timing breakdown (per-query averages) ===
  Stage                          ms / query
  ──────────────────────────── ────────────
  HNSW probe                         133.26
  Exact probe                          3.55
    └─ total probe                   136.81
  Prune (alpha+eps)                    1.87
  Rerank (exact WJ)                  132.67
  Total per query                    295.35

=== Candidate stats (per-query averages) ===
  HNSW buckets hit                    738.5
  Exact buckets hit                  1097.8
  HNSW candidates added         

In [10]:
bp.log_result(
    notebook="v1.16_HNSW",
    backend="hnsw",
    dataset=_dataset_key,
    n_db=int(DATA_END),
    n_query=int(QUERY_END - QUERY_START),
    D=int(D),
    L=int(L),
    B=int(B),
    R=int(R),
    tau_min_size=int(BUCKET_INDEX_MIN_SIZE),
    split_thresh=int(REB_SPLIT_LARGE),
    index_metric="WeightedJaccard",
    backend_params=f"M{LOCAL_HNSW_M}_efc{LOCAL_HNSW_EFC}_efs{LOCAL_HNSW_EFS}",
    alpha_mult=FAST_ALPHA_MULT,
    epsilon=int(FAST_EPSILON),
    efS=int(FAST_EFS),
    b_max=MAX_BANDS_TO_PROBE,
    stage="rerank",
    r10=recall.get(10),
    r50=recall.get(50),
    r500=recall.get(500),
    qps=summary["qps"],
    build_time_sec=build_stats.get("build_time_sec"),
    source="v1.16",
)


[log_result] → /raid/ruban/boi_multi_index/results.csv
  v1.16_HNSW | hnsw | rerank | R@10=0.9885 R@50=0.9821 R@500=0.8869 | QPS=261.48
